# Reciprocal Rank Fusion (RRF) with Position-Aware Blending

In the previous notebook, we used LLM-based query expansion to generate alternative phrasings. Now we'll put that to work - generating multiple query variations and combining their results using Reciprocal Rank Fusion.

Reciprocal Rank Fusion is an elegant method for combining multiple ranked lists without needing to normalize scores. It's based on a simple insight: the rank of a document matters more than its raw score.

In this notebook, we'll implement a complete RRF-based retrieval pipeline including:
1. RRF fusion of multiple ranked lists
2. Query expansion with parallel retrieval
3. LLM-based reranking
4. Position-aware blending

## The RRF Formula

For a document `d` appearing in multiple ranked lists, the RRF score is:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

Where:
- `R` is the set of ranked lists
- `rank_r(d)` is the rank of document `d` in list `r` (1-indexed)
- `k` is a constant (typically 60) that prevents high scores from dominating

**In plain English:** For each document, look at where it ranks in every search result list. For each list, compute 1 divided by (k + the document's rank in that list), then add all those values together. A document that ranks highly in many lists accumulates a high total score. The constant `k` (usually 60) acts as a dampener -- without it, a rank-1 result would get a score of 1.0 and completely overpower everything else. With k=60, rank 1 scores 1/61 and rank 2 scores 1/62, so the gap between positions is small and gradual. This means a document that appears at rank 5 in three different lists can outscore a document that appears at rank 1 in only one list -- which is exactly the behavior we want when fusing results.

The key insight: RRF uses **ranks**, not scores. This sidesteps the normalization problem entirely.

In [ ]:
# DEPENDENCY: pip install -q openai numpy
# (packages should be pre-installed in venv before running this script)

In [ ]:
import os
import json
import math
import numpy as np
from getpass import getpass
from collections import Counter, defaultdict
from dataclasses import dataclass, field
import re
from openai import OpenAI

# API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

client = OpenAI()
MODEL = "gpt-5-mini"
EMBED_MODEL = "text-embedding-3-small"

## Part 1: Basic RRF Implementation

Let's start with a pure RRF implementation.

In [ ]:
def reciprocal_rank_fusion(
    ranked_lists: list[list[int]],
    k: int = 60
) -> list[tuple[int, float]]:
    """
    Combine multiple ranked lists using Reciprocal Rank Fusion.
    
    Args:
        ranked_lists: List of ranked lists (each list contains doc indices in rank order)
        k: Smoothing constant (default 60)
    
    Returns:
        List of (doc_idx, rrf_score) tuples sorted by score descending
    """
    rrf_scores = defaultdict(float)
    
    for ranked_list in ranked_lists:
        for rank, doc_idx in enumerate(ranked_list, start=1):
            rrf_scores[doc_idx] += 1.0 / (k + rank)
    
    # Sort by RRF score descending
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results

# Example: Fuse three ranked lists
list1 = [0, 2, 4, 1, 3]  # Rankings from BM25
list2 = [2, 0, 1, 4, 3]  # Rankings from vector search
list3 = [0, 1, 2, 3, 4]  # Rankings from another method

fused = reciprocal_rank_fusion([list1, list2, list3])

print("Individual rankings:")
print(f"  List 1 (BM25):   {list1}")
print(f"  List 2 (Vector): {list2}")
print(f"  List 3 (Other):  {list3}")
print(f"\nFused RRF ranking:")
for doc_idx, score in fused:
    print(f"  Doc {doc_idx}: RRF score = {score:.6f}")

## Understanding RRF Properties

Let's visualize how RRF behaves with different k values and ranking positions.

In [ ]:
def rrf_contribution(rank: int, k: int = 60) -> float:
    """Calculate RRF contribution for a given rank."""
    return 1.0 / (k + rank)

# Show contribution decay
print("RRF contribution by rank (k=60):")
print("-" * 40)
ranks = [1, 2, 3, 5, 10, 20, 50, 100]
for rank in ranks:
    contrib = rrf_contribution(rank)
    bar = "█" * int(contrib * 1000)
    print(f"Rank {rank:3d}: {contrib:.6f} {bar}")

print("\n" + "=" * 50)
print("\nEffect of k parameter:")
print("-" * 40)

k_values = [10, 30, 60, 100]
print(f"{'Rank':<8}", end="")
for k in k_values:
    print(f"k={k:<6}", end="")
print()

for rank in [1, 5, 10, 50]:
    print(f"{rank:<8}", end="")
    for k in k_values:
        print(f"{rrf_contribution(rank, k):.4f}  ", end="")
    print()

## Part 2: Weighted RRF

In production systems, not all ranked lists are equally trustworthy. We can weight lists differently.

In [ ]:
def weighted_rrf(
    ranked_lists: list[list[int]],
    weights: list[float] = None,
    k: int = 60
) -> list[tuple[int, float]]:
    """
    RRF with optional weights for each ranked list.
    
    Example use case:
    - Original query results: weight = 2.0
    - Expanded query results: weight = 1.0
    """
    if weights is None:
        weights = [1.0] * len(ranked_lists)
    
    rrf_scores = defaultdict(float)
    
    for weight, ranked_list in zip(weights, ranked_lists):
        for rank, doc_idx in enumerate(ranked_list, start=1):
            rrf_scores[doc_idx] += weight / (k + rank)
    
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results

# Compare weighted vs unweighted
original_query_results = [0, 2, 4]
expanded_query_results = [3, 2, 1]

print("Unweighted RRF:")
unweighted = reciprocal_rank_fusion([original_query_results, expanded_query_results])
for doc, score in unweighted[:5]:
    print(f"  Doc {doc}: {score:.6f}")

print("\nWeighted RRF (original query 2x):")
weighted = weighted_rrf([original_query_results, expanded_query_results], weights=[2.0, 1.0])
for doc, score in weighted[:5]:
    print(f"  Doc {doc}: {score:.6f}")

## Part 3: Top-Rank Bonus

Documents that rank #1 in any list deserve extra credit - they're likely highly relevant for at least one interpretation of the query.

In [ ]:
def rrf_with_top_rank_bonus(
    ranked_lists: list[list[int]],
    k: int = 60,
    bonus_rank_1: float = 0.05,
    bonus_rank_2_3: float = 0.02
) -> list[tuple[int, float]]:
    """
    RRF with bonus for documents ranking in top positions.
    
    This preserves high-confidence matches that might otherwise
    be diluted by expanded queries.
    """
    rrf_scores = defaultdict(float)
    bonuses = defaultdict(float)
    
    for ranked_list in ranked_lists:
        for rank, doc_idx in enumerate(ranked_list, start=1):
            # Standard RRF contribution
            rrf_scores[doc_idx] += 1.0 / (k + rank)
            
            # Track top-rank bonuses (keep highest)
            if rank == 1:
                bonuses[doc_idx] = max(bonuses[doc_idx], bonus_rank_1)
            elif rank in [2, 3]:
                bonuses[doc_idx] = max(bonuses[doc_idx], bonus_rank_2_3)
    
    # Apply bonuses
    final_scores = {}
    for doc_idx, rrf_score in rrf_scores.items():
        final_scores[doc_idx] = rrf_score + bonuses.get(doc_idx, 0)
    
    sorted_results = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results

# Example showing bonus effect
lists = [
    [5, 2, 3, 1, 4],  # Doc 5 is #1 here
    [1, 3, 5, 2, 4],  # Doc 1 is #1 here
    [2, 1, 3, 4, 5],  # Doc 2 is #1 here
]

print("Without top-rank bonus:")
standard = reciprocal_rank_fusion(lists)
for doc, score in standard[:5]:
    print(f"  Doc {doc}: {score:.6f}")

print("\nWith top-rank bonus (+0.05 for #1, +0.02 for #2-3):")
with_bonus = rrf_with_top_rank_bonus(lists)
for doc, score in with_bonus[:5]:
    print(f"  Doc {doc}: {score:.6f}")

## Part 4: LLM Reranking

After RRF fusion, we take the top candidates and rerank them using an LLM. This adds semantic understanding to the ranking.

In [ ]:
def rerank_with_llm(
    query: str,
    documents: list[str],
    doc_indices: list[int]
) -> list[tuple[int, float]]:
    """
    Rerank documents using LLM relevance scoring.
    
    Returns list of (doc_idx, relevance_score) tuples.
    Score is 0.0-1.0 based on LLM assessment.
    """
    results = []
    
    for doc_idx in doc_indices:
        doc_text = documents[doc_idx][:500]  # Truncate for efficiency
        
        prompt = f"""Rate the relevance of this document to the query.

Query: {query}
Document: {doc_text}

Is this document relevant to the query? Answer with just "yes" or "no", then a confidence score from 0-10.
Format: yes/no, score
Example: yes, 8"""
        
        response = client.responses.create(
            model=MODEL,
            input=prompt
        )
        
        # Parse response
        text = response.output_text.lower().strip()
        try:
            parts = text.split(",")
            is_relevant = "yes" in parts[0]
            score = int(parts[1].strip()) / 10.0
            
            # If not relevant, reduce score
            if not is_relevant:
                score *= 0.3
                
            results.append((doc_idx, score))
        except:
            # Fallback if parsing fails
            results.append((doc_idx, 0.5))
    
    return results

# Test reranking
test_docs = [
    "Python is a programming language known for readability.",
    "Machine learning uses algorithms to find patterns in data.",
    "The Python snake is found in tropical regions.",
    "Deep learning neural networks require large datasets.",
    "JavaScript is used for web development.",
]

query = "Python programming tutorials"
candidates = [0, 1, 2, 3, 4]  # All docs as candidates

print(f"Query: '{query}'")
print("\nLLM Reranking scores:")
reranked = rerank_with_llm(query, test_docs, candidates)
reranked.sort(key=lambda x: x[1], reverse=True)

for doc_idx, score in reranked:
    print(f"  [{score:.2f}] Doc {doc_idx}: {test_docs[doc_idx][:50]}...")

## Part 5: Position-Aware Blending

The final piece: blend RRF scores with reranker scores, but vary the blend based on the RRF rank.

The intuition:
- **Top RRF ranks (1-3)**: These are likely exact matches. Trust retrieval more (75% RRF, 25% reranker)
- **Middle RRF ranks (4-10)**: Mixed signals. Use balanced blend (60% RRF, 40% reranker)
- **Lower RRF ranks (11+)**: Less confident retrieval. Trust reranker more (40% RRF, 60% reranker)

In [ ]:
def position_aware_blend(
    rrf_results: list[tuple[int, float]],
    rerank_results: list[tuple[int, float]]
) -> list[tuple[int, float, dict]]:
    """
    Blend RRF and reranker scores with position-aware weighting.
    
    Returns:
        List of (doc_idx, final_score, breakdown) tuples
    """
    # Create lookup dictionaries
    rrf_by_doc = {doc: (rank, score) for rank, (doc, score) in enumerate(rrf_results, 1)}
    rerank_by_doc = {doc: score for doc, score in rerank_results}
    
    # Normalize RRF scores to 0-1
    max_rrf = max(score for _, score in rrf_results) if rrf_results else 1
    
    results = []
    for doc_idx, rrf_score in rrf_results:
        rrf_rank = rrf_by_doc[doc_idx][0]
        rrf_normalized = rrf_score / max_rrf
        rerank_score = rerank_by_doc.get(doc_idx, 0.5)
        
        # Position-aware blending weights
        if rrf_rank <= 3:
            rrf_weight = 0.75
            blend_type = "top"
        elif rrf_rank <= 10:
            rrf_weight = 0.60
            blend_type = "mid"
        else:
            rrf_weight = 0.40
            blend_type = "low"
        
        rerank_weight = 1 - rrf_weight
        
        # Compute blended score
        final_score = rrf_weight * rrf_normalized + rerank_weight * rerank_score
        
        results.append((doc_idx, final_score, {
            "rrf_rank": rrf_rank,
            "rrf_score": rrf_normalized,
            "rerank_score": rerank_score,
            "blend_type": blend_type,
            "rrf_weight": rrf_weight
        }))
    
    # Sort by final score
    results.sort(key=lambda x: x[1], reverse=True)
    return results

## Part 6: Complete Pipeline

Now let's put it all together into a complete retrieval pipeline.

In [ ]:
@dataclass
class AdvancedRetriever:
    """
    Complete retrieval pipeline with:
    1. Query expansion
    2. Parallel BM25 + Vector retrieval
    3. RRF fusion with top-rank bonus
    4. LLM reranking
    5. Position-aware blending
    """
    k: int = 60
    top_k_candidates: int = 30
    
    def __post_init__(self):
        self.documents = []
        self.bm25_index = None
        self.embeddings = []
    
    def index(self, documents: list[str]):
        """Index documents for retrieval."""
        self.documents = documents
        
        # BM25 indexing (simplified - reuse our implementation)
        self.bm25_index = self._build_bm25_index(documents)
        
        # Vector indexing
        print("Computing embeddings...")
        response = client.embeddings.create(
            model=EMBED_MODEL,
            input=documents
        )
        self.embeddings = [item.embedding for item in response.data]
        print(f"Indexed {len(documents)} documents")
    
    def _build_bm25_index(self, documents):
        """Build BM25 index."""
        # Tokenize
        def tokenize(text):
            return [t.lower() for t in re.findall(r'\b\w+\b', text) if len(t) > 1]
        
        doc_freqs = [Counter(tokenize(doc)) for doc in documents]
        doc_lengths = [sum(freq.values()) for freq in doc_freqs]
        avgdl = sum(doc_lengths) / len(doc_lengths)
        
        # IDF
        doc_containing = Counter()
        for freq in doc_freqs:
            for term in freq:
                doc_containing[term] += 1
        
        n = len(documents)
        idf = {term: math.log((n - df + 0.5) / (df + 0.5) + 1) 
               for term, df in doc_containing.items()}
        
        return {"doc_freqs": doc_freqs, "doc_lengths": doc_lengths, 
                "avgdl": avgdl, "idf": idf, "tokenize": tokenize}
    
    def _bm25_search(self, query: str, top_k: int = 100) -> list[int]:
        """BM25 search returning ranked doc indices."""
        idx = self.bm25_index
        query_terms = idx["tokenize"](query)
        k1, b = 1.5, 0.75
        
        scores = []
        for i, doc_freq in enumerate(idx["doc_freqs"]):
            score = 0
            for term in query_terms:
                if term in idx["idf"]:
                    tf = doc_freq.get(term, 0)
                    idf = idx["idf"][term]
                    dl = idx["doc_lengths"][i]
                    score += idf * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / idx["avgdl"]))
            scores.append((i, score))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return [i for i, s in scores[:top_k] if s > 0]
    
    def _vector_search(self, query: str, top_k: int = 100) -> list[int]:
        """Vector search returning ranked doc indices."""
        response = client.embeddings.create(model=EMBED_MODEL, input=query)
        query_emb = np.array(response.data[0].embedding)
        
        scores = []
        for i, doc_emb in enumerate(self.embeddings):
            doc_emb = np.array(doc_emb)
            sim = np.dot(query_emb, doc_emb) / (np.linalg.norm(query_emb) * np.linalg.norm(doc_emb))
            scores.append((i, sim))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return [i for i, _ in scores[:top_k]]
    
    def _expand_query(self, query: str) -> list[str]:
        """Generate query expansions."""
        prompt = f"""Generate 2 alternative search queries for: {query}
Return only a JSON array of strings."""
        
        response = client.responses.create(model=MODEL, input=prompt)
        try:
            return json.loads(response.output_text)
        except:
            return []
    
    def search(self, query: str, top_k: int = 10, verbose: bool = False) -> list[dict]:
        """Full retrieval pipeline."""
        
        # Step 1: Query expansion
        queries = [query] + self._expand_query(query)
        if verbose:
            print(f"Queries: {queries}")
        
        # Step 2: Parallel retrieval
        ranked_lists = []
        weights = []
        
        for i, q in enumerate(queries):
            # BM25
            bm25_results = self._bm25_search(q)
            ranked_lists.append(bm25_results)
            weights.append(2.0 if i == 0 else 1.0)  # Original query gets 2x weight
            
            # Vector
            vector_results = self._vector_search(q)
            ranked_lists.append(vector_results)
            weights.append(2.0 if i == 0 else 1.0)
        
        # Step 3: RRF fusion with top-rank bonus
        rrf_scores = defaultdict(float)
        bonuses = defaultdict(float)
        
        for weight, ranked_list in zip(weights, ranked_lists):
            for rank, doc_idx in enumerate(ranked_list, start=1):
                rrf_scores[doc_idx] += weight / (self.k + rank)
                if rank == 1:
                    bonuses[doc_idx] = max(bonuses[doc_idx], 0.05)
                elif rank in [2, 3]:
                    bonuses[doc_idx] = max(bonuses[doc_idx], 0.02)
        
        # Apply bonuses and sort
        rrf_results = [(doc, score + bonuses.get(doc, 0)) 
                       for doc, score in rrf_scores.items()]
        rrf_results.sort(key=lambda x: x[1], reverse=True)
        
        # Step 4: Take top candidates for reranking
        candidates = [doc for doc, _ in rrf_results[:self.top_k_candidates]]
        
        if verbose:
            print(f"RRF top candidates: {candidates[:10]}")
        
        # Step 5: LLM reranking
        rerank_results = rerank_with_llm(query, self.documents, candidates)
        
        # Step 6: Position-aware blending
        blended = position_aware_blend(rrf_results[:self.top_k_candidates], rerank_results)
        
        # Format results
        final_results = []
        for doc_idx, score, breakdown in blended[:top_k]:
            final_results.append({
                "doc_idx": doc_idx,
                "score": score,
                "text": self.documents[doc_idx],
                "breakdown": breakdown
            })
        
        return final_results

In [ ]:
# Test the complete pipeline
corpus = [
    "Python is a high-level programming language emphasizing code readability and simplicity.",
    "Machine learning enables computers to learn from data without explicit programming.",
    "Deep learning uses neural networks with many layers to model complex patterns.",
    "Natural language processing helps computers understand and generate human language.",
    "SQL is a domain-specific language for managing relational databases.",
    "NoSQL databases provide flexible schemas for unstructured data storage.",
    "Docker containers package applications with dependencies for deployment.",
    "Kubernetes orchestrates containerized applications across clusters.",
    "Git enables version control and collaboration on source code.",
    "Agile methodology focuses on iterative development and customer feedback.",
    "REST APIs use HTTP methods for communication between systems.",
    "GraphQL provides a query language for APIs with strong typing.",
]

retriever = AdvancedRetriever()
retriever.index(corpus)

In [ ]:
# Run a search
query = "how to store and query data"
results = retriever.search(query, top_k=5, verbose=True)

print(f"\nResults for: '{query}'")
print("=" * 70)

for i, result in enumerate(results, 1):
    b = result["breakdown"]
    print(f"\n{i}. [Score: {result['score']:.3f}] (RRF rank: {b['rrf_rank']}, blend: {b['blend_type']})")
    print(f"   RRF: {b['rrf_score']:.3f} × {b['rrf_weight']:.0%} + Rerank: {b['rerank_score']:.3f} × {1-b['rrf_weight']:.0%}")
    print(f"   {result['text'][:80]}...")

---

## Exercises

### Exercise 1: Adaptive k Parameter

The k parameter in RRF affects score distribution. Implement an adaptive k that changes based on the number of ranked lists or query characteristics.

In [ ]:
def adaptive_k(num_lists: int, avg_list_length: int) -> int:
    """Compute adaptive k based on fusion characteristics."""
    # Your implementation here
    pass

### Exercise 2: Confidence-Based Reranking

Extend the reranker to return a confidence score along with relevance. Use confidence to weight the reranker's contribution to the final blend.

In [ ]:
def confidence_weighted_blend(
    rrf_results: list[tuple[int, float]],
    rerank_results: list[tuple[int, float, float]]  # (doc, score, confidence)
) -> list[tuple[int, float]]:
    """Blend with confidence-weighted reranker scores."""
    # Your implementation here
    pass

### Exercise 3: Cascaded Reranking

Implement a two-stage reranker:
1. Fast reranker (small model) to filter candidates
2. Slow reranker (large model) for final ranking

In [ ]:
def cascaded_rerank(
    query: str,
    documents: list[str],
    candidates: list[int],
    stage1_keep: int = 10
) -> list[tuple[int, float]]:
    """Two-stage cascaded reranking."""
    # Your implementation here
    pass

---

## Summary

We implemented a complete retrieval pipeline following the QMD architecture:

1. **Query Expansion**: Original query (x2 weight) + LLM variations
2. **Parallel Retrieval**: Each query runs both BM25 and vector search
3. **RRF Fusion**: Combine all lists using `score = Σ(weight/(k+rank))`
4. **Top-Rank Bonus**: +0.05 for #1, +0.02 for #2-3 in any list
5. **LLM Reranking**: Score top 30 candidates for relevance
6. **Position-Aware Blending**:
   - Ranks 1-3: 75% RRF, 25% reranker
   - Ranks 4-10: 60% RRF, 40% reranker
   - Ranks 11+: 40% RRF, 60% reranker

**In plain English:** The fusion formula `score = Σ(weight/(k+rank))` works the same way as the basic RRF formula, but now each list gets a weight multiplier. The original query's results get double the weight (2.0) compared to expanded queries (1.0), so the original user intent always has the strongest voice. After fusion, we layer on a small bonus for documents that ranked #1 or #2-3 in any single list -- this preserves high-confidence matches that might otherwise get diluted when many expanded queries are fused together.

This approach balances:
- **Precision** (exact matches via BM25, top-rank bonus)
- **Recall** (semantic via embeddings, query expansion)
- **Relevance** (LLM reranking with position-aware trust)

In the next notebook, we'll explore HyDE (Hypothetical Document Embeddings) and Parent Document Retrieval - two techniques that improve RAG quality by fixing the query-document mismatch and the chunking trade-off.